In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, hashlib, cv2
import numpy as np
from glob import glob
from tqdm import tqdm
from collections import defaultdict
import re

DATASET_DIR = "/content/drive/MyDrive/Project work/Dataset/brisc2025/segmentation_task"
#G:\My Drive\Project work\Dataset\brisc2025\segmentation_task

# ================================
# LOAD DATA
# ================================
def load_all():
    imgs, masks = [], []
    for split in ["train", "test"]:
        imgs  += glob(os.path.join(DATASET_DIR, split, "images", "*"))
        masks += glob(os.path.join(DATASET_DIR, split, "masks", "*"))
    return sorted(imgs), sorted(masks)

images, masks = load_all()

print(f"Total images: {len(images)}")
print(f"Total masks : {len(masks)}")

assert len(images) == len(masks), "❌ Image-mask count mismatch"

# ================================
# EXACT DUPLICATE CHECK (HASH)
# ================================
def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hash_map = {}
duplicate_count = 0

for img in tqdm(images, desc="Checking exact duplicates"):
    h = file_hash(img)
    if h in hash_map:
        duplicate_count += 1
    else:
        hash_map[h] = img

print(f"\nExact duplicate images found: {duplicate_count}")

# ================================
# NEAR DUPLICATE CHECK
# ================================
def signature(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    return img.flatten()

signatures = []
near_duplicates = 0

THRESHOLD = 5

for img in tqdm(images[:2000], desc="Checking near duplicates (sample)"):
    sig = signature(img)

    for s in signatures:
        if np.mean(np.abs(sig - s)) < THRESHOLD:
            near_duplicates += 1
            break

    signatures.append(sig)

print(f"Near duplicates (sample of 2000): {near_duplicates}")

# ================================
# PATIENT ID CHECK (CRITICAL)
# ================================
def get_patient_id(path):
    name = os.path.basename(path)
    return name.split("_")[0]   # ⚠️ VERIFY THIS

print("\nSample patient IDs:")
for i in range(5):
    print(os.path.basename(images[i]), "→", get_patient_id(images[i]))

# ================================
# LEAKAGE CHECK (TRAIN vs TEST)
# ================================
train_imgs = glob(os.path.join(DATASET_DIR, "train/images/*"))
test_imgs  = glob(os.path.join(DATASET_DIR, "test/images/*"))

train_patients = set(get_patient_id(p) for p in train_imgs)
test_patients  = set(get_patient_id(p) for p in test_imgs)

leak = train_patients & test_patients

print("\nLeakage check:")
print(f"Train patients: {len(train_patients)}")
print(f"Test patients : {len(test_patients)}")
print(f"Overlap       : {len(leak)}")

if len(leak) > 0:
    print("❌ DATA LEAKAGE DETECTED")
else:
    print("✅ No leakage")

Total images: 4793
Total masks : 4793


Checking exact duplicates: 100%|██████████| 4793/4793 [40:15<00:00,  1.98it/s]



Exact duplicate images found: 40


Checking near duplicates (sample): 100%|██████████| 2000/2000 [00:51<00:00, 38.58it/s]


Near duplicates (sample of 2000): 7

Sample patient IDs:
brisc2025_test_00001_gl_ax_t1.jpg → brisc2025
brisc2025_test_00002_gl_ax_t1.jpg → brisc2025
brisc2025_test_00003_gl_ax_t1.jpg → brisc2025
brisc2025_test_00004_gl_ax_t1.jpg → brisc2025
brisc2025_test_00005_gl_ax_t1.jpg → brisc2025

Leakage check:
Train patients: 1
Test patients : 1
Overlap       : 1
❌ DATA LEAKAGE DETECTED


In [ ]:
import os, re, hashlib, cv2
import numpy as np
from glob import glob
from tqdm import tqdm

# =====================================
# LOAD DATA
# =====================================
def load_split(split):
    imgs  = glob(os.path.join(DATASET_DIR, split, "images", "*"))
    masks = glob(os.path.join(DATASET_DIR, split, "masks", "*"))
    return sorted(imgs), sorted(masks)

train_imgs, train_masks = load_split("train")
test_imgs,  test_masks  = load_split("test")

all_images = sorted(train_imgs + test_imgs)
all_masks  = sorted(train_masks + test_masks)

print("Total images:", len(all_images))
print("Total masks :", len(all_masks))

assert len(all_images) == len(all_masks), "❌ mismatch"

# =====================================
# EXACT DUPLICATE CHECK
# =====================================
def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

seen = set()
dup_count = 0

for img in tqdm(all_images, desc="Exact duplicate check"):
    h = file_hash(img)
    if h in seen:
        dup_count += 1
    else:
        seen.add(h)

print("\nExact duplicates:", dup_count)

# =====================================
# NEAR DUPLICATE CHECK (SAMPLE)
# =====================================
def signature(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    return img.flatten()

sigs = []
near_dup = 0
THRESHOLD = 5

for img in tqdm(all_images[:2000], desc="Near duplicate check"):
    sig = signature(img)

    for s in sigs:
        if np.mean(np.abs(sig - s)) < THRESHOLD:
            near_dup += 1
            break

    sigs.append(sig)

print("Near duplicates (sample):", near_dup)

# =====================================
# CORRECT PATIENT ID EXTRACTION
# =====================================
def get_patient_id(path):
    name = os.path.basename(path)
    match = re.search(r"\d+", name)
    return match.group(0) if match else name

print("\nSample patient IDs:")
for i in range(5):
    print(os.path.basename(all_images[i]), "→", get_patient_id(all_images[i]))

# =====================================
# LEAKAGE CHECK
# =====================================
train_patients = set(get_patient_id(p) for p in train_imgs)
test_patients  = set(get_patient_id(p) for p in test_imgs)

overlap = train_patients & test_patients

print("\n===== LEAKAGE REPORT =====")
print("Train patients :", len(train_patients))
print("Test patients  :", len(test_patients))
print("Leak count     :", len(overlap))

if len(overlap) > 0:
    print("\nLeaking patient IDs (first 20):")
    print(list(overlap)[:20])
    print("❌ DATA LEAKAGE PRESENT")
else:
    print("✅ No leakage")

# =====================================
# EXTRA: IMAGE-LEVEL LEAKAGE (OPTIONAL)
# =====================================
train_hash = set(file_hash(p) for p in train_imgs)
test_hash  = set(file_hash(p) for p in test_imgs)

img_overlap = train_hash & test_hash

print("\nImage-level leakage (exact same images in train & test):", len(img_overlap))

Total images: 4793
Total masks : 4793


Exact duplicate check: 100%|██████████| 4793/4793 [00:22<00:00, 211.83it/s]



Exact duplicates: 40


Near duplicate check: 100%|██████████| 2000/2000 [00:52<00:00, 38.09it/s]


Near duplicates (sample): 7

Sample patient IDs:
brisc2025_test_00001_gl_ax_t1.jpg → 2025
brisc2025_test_00002_gl_ax_t1.jpg → 2025
brisc2025_test_00003_gl_ax_t1.jpg → 2025
brisc2025_test_00004_gl_ax_t1.jpg → 2025
brisc2025_test_00005_gl_ax_t1.jpg → 2025

===== LEAKAGE REPORT =====
Train patients : 1
Test patients  : 1
Leak count     : 1

Leaking patient IDs (first 20):
['2025']
❌ DATA LEAKAGE PRESENT

Image-level leakage (exact same images in train & test): 7


In [ ]:
def get_patient_id(path):
    name = os.path.basename(path)

    # Extract the 5-digit ID after "test_" or "train_"
    match = re.search(r"(?:train|test)_(\d+)", name)

    if match:
        return match.group(1)
    else:
        return name

In [ ]:
train_patients = set(get_patient_id(p) for p in train_imgs)
test_patients  = set(get_patient_id(p) for p in test_imgs)

overlap = train_patients & test_patients

print("Train patients:", len(train_patients))
print("Test patients :", len(test_patients))
print("Leak count    :", len(overlap))
print("Sample leaks  :", list(overlap)[:10])

Train patients: 3933
Test patients : 860
Leak count    : 860
Sample leaks  : ['00360', '00487', '00881', '00531', '00235', '00801', '00821', '00006', '00893', '00335']


In [ ]:
import os, shutil, random, re, json
from glob import glob
from collections import defaultdict
from tqdm import tqdm

# =========================
# CONFIG (EDIT THIS ONLY)
# =========================
OUTPUT_DIR  = "/content/drive/MyDrive/Project work/Dataset/brisc2025/segmentation_task_ttv"

TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1
SEED = 42

random.seed(SEED)

# =========================
# LOAD ALL DATA (MERGE)
# =========================
def load_all():
    imgs, masks = [], []
    for split in ["train", "test"]:
        imgs  += glob(os.path.join(DATASET_DIR, split, "images", "*"))
        masks += glob(os.path.join(DATASET_DIR, split, "masks", "*"))
    return sorted(imgs), sorted(masks)

images, masks = load_all()

print("Total images:", len(images))
assert len(images) == len(masks), "Mismatch"

# =========================
# CORRECT PATIENT ID
# =========================
def get_patient_id(path):
    name = os.path.basename(path)
    match = re.search(r"(?:train|test)_(\d+)", name)
    return match.group(1) if match else name

# verify
print("\nSample mapping:")
for i in range(5):
    print(os.path.basename(images[i]), "→", get_patient_id(images[i]))

# =========================
# GROUP BY PATIENT
# =========================
patient_dict = defaultdict(list)

for img, mask in zip(images, masks):
    pid = get_patient_id(img)
    patient_dict[pid].append((img, mask))

patients = list(patient_dict.keys())
print("\nTotal patients:", len(patients))

# =========================
# SPLIT (NO LEAKAGE)
# =========================
random.shuffle(patients)

n = len(patients)
train_end = int(n * TRAIN_RATIO)
val_end   = train_end + int(n * VAL_RATIO)

train_p = patients[:train_end]
val_p   = patients[train_end:val_end]
test_p  = patients[val_end:]

print("\nSplit:")
print("Train:", len(train_p))
print("Val  :", len(val_p))
print("Test :", len(test_p))

# =========================
# CREATE FOLDERS
# =========================
for split in ["train", "val", "test"]:
    for t in ["images", "masks"]:
        os.makedirs(os.path.join(OUTPUT_DIR, split, t), exist_ok=True)

# =========================
# COPY DATA
# =========================
def copy_split(patients, split):
    count = 0
    for p in tqdm(patients, desc=f"Copying {split}"):
        for img, mask in patient_dict[p]:
            shutil.copy2(img, os.path.join(OUTPUT_DIR, split, "images"))
            shutil.copy2(mask, os.path.join(OUTPUT_DIR, split, "masks"))
            count += 1
    print(f"{split}: {count} samples")

copy_split(train_p, "train")
copy_split(val_p,   "val")
copy_split(test_p,  "test")

# =========================
# SAVE SPLIT RECORD
# =========================
split_record = {
    "train": train_p,
    "val": val_p,
    "test": test_p
}

with open(os.path.join(OUTPUT_DIR, "split.json"), "w") as f:
    json.dump(split_record, f, indent=4)

# =========================
# FINAL LEAKAGE CHECK
# =========================
print("\nLeakage check:")
print("Train ∩ Val :", len(set(train_p) & set(val_p)))
print("Train ∩ Test:", len(set(train_p) & set(test_p)))
print("Val ∩ Test  :", len(set(val_p) & set(test_p)))

print("\n✅ Done — dataset is clean and ready")

Total images: 4793

Sample mapping:
brisc2025_test_00001_gl_ax_t1.jpg → 00001
brisc2025_test_00002_gl_ax_t1.jpg → 00002
brisc2025_test_00003_gl_ax_t1.jpg → 00003
brisc2025_test_00004_gl_ax_t1.jpg → 00004
brisc2025_test_00005_gl_ax_t1.jpg → 00005

Total patients: 3933

Split:
Train: 3146
Val  : 393
Test : 394


Copying train: 100%|██████████| 3146/3146 [45:54<00:00,  1.14it/s]


train: 3837 samples


Copying val: 100%|██████████| 393/393 [05:34<00:00,  1.18it/s]


val: 477 samples


Copying test: 100%|██████████| 394/394 [05:30<00:00,  1.19it/s]

test: 479 samples

Leakage check:
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

✅ Done — dataset is clean and ready
